# Weighted Least Squares: the idea in plain language

Weighted Least Squares (WLS) fits a model to measurements while taking their reliability into account. Ordinary least squares treats every measurement equally. WLS gives **more influence to accurate measurements** and **less influence to noisy measurements**.

## The measurement model

We assume that each measurement can be described by

$$
\mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{v}
$$

- \(\mathbf{y}\): the measurements we observed.
- \(\mathbf{H}\): the model matrix, which describes how the unknowns affect each measurement.
- \(\mathbf{x}\): the unknown parameters we want to estimate.
- \(\mathbf{v}\): measurement error or noise.

The covariance matrix \(\mathbf{R}\) describes the uncertainty of the measurements. For independent measurements, it is diagonal:

$$
\mathbf{R} = \operatorname{diag}(\sigma_1^2, \sigma_2^2, \ldots, \sigma_n^2)
$$

Here \(\sigma_i\) is the standard deviation of measurement \(i\). A large \(\sigma_i\) means that measurement is less certain. WLS uses \(\mathbf{R}^{-1}\) as the weighting matrix, so the weight of an independent measurement is

$$
w_i = \frac{1}{\sigma_i^2}.
$$

For example, a measurement with \(\sigma=1\) has weight \(1\), while one with \(\sigma=2\) has weight \(1/4\). The second measurement is therefore treated as four times less reliable.

## The WLS estimate

WLS chooses the parameter vector that best fits the measurements after accounting for their uncertainties:

$$
\hat{\mathbf{x}} = (\mathbf{H}^T\mathbf{R}^{-1}\mathbf{H})^{-1}\mathbf{H}^T\mathbf{R}^{-1}\mathbf{y}.
$$

The covariance of this estimate is

$$
\mathbf{P} = (\mathbf{H}^T\mathbf{R}^{-1}\mathbf{H})^{-1}.
$$

In this notebook, the model is a straight line relating temperature \(y\) to RPM \(r\):

$$\hat{y} = \hat{x}_1 r + \hat{x}_2,$$

where \(\hat{x}_1\) is the estimated slope and \(\hat{x}_2\) is the estimated intercept. Each dataset row is stored as \([y_i, r_i, \sigma_i]\). The next section derives the corresponding \(\mathbf{y}\), \(\mathbf{H}\), and \(\mathbf{R}\) matrices.

## Mathematical Explanation

In linear least squares estimation, we want to find a parameter vector $x$ that best fits a linear model of the form:

$$y = Hx + v$$

Where:

- $y$ is the measurement vector (dependent variable, e.g., temperature).
- $H$ is the model/design matrix (independent variables/basis functions, e.g., RPM values and a column of ones for the intercept).
- $x$ is the constant vector of parameters we want to estimate (e.g., slope $x_1$ and intercept $x_2$).
- $v$ is the measurement noise.

The analytical solution that minimizes the sum of the squared residuals is given by the normal equations:

$$x = (H^T H)^{-1} H^T y$$

Using the hints provided, this can be implemented in NumPy using np.matmul(), np.transpose(), and np.linalg.inv().

For the line of best fit ($y_i = x_1 \cdot r_i + x_2$), each data point provides a row in our system:

- The model matrix $H$ has two columns: the first column contains the RPM values ($r_i$), and the second column contains ones (for the intercept $x_2$).
- The measurement vector $Y$ contains the temperature values ($y_i$).
---

In [3]:
import numpy as np

Dataset = [
    [62, 1, 1],
    [68, 2, 2],
    [74, 3, 1.5],
    [82, 4, 1],
    [89, 5, 2],
    [95, 6, 1],
    [103, 7, 1.5],
    [109, 8, 2]
]

In [4]:
def CalculateLeastSquaresSolution(HMatrix, YMatrix):
    """
    Calculates the general LSE solution x given H and Y
    Formula: x = (H^T * H)^-1 * H^T * Y
    """
    # Convert inputs to numpy arrays to ensure matrix operations
    H = np.array(HMatrix)
    Y = np.array(YMatrix)

    # Transpose of H Matrix 
    H_T = np.transpose(H)

    # Calculate (H^T, H)
    HtH = np.malmut(H_T,H)

    # Calculate the inverse of (H^T * H)^-1
    HtH_inverse = np.linalg.inv(HtH)

    # Calculate (H^T * Y)
    HtY = np.malmut(H_T,Y)

    # Calculate the final parameter vector X = (H^T * H)^-1*H^T*Y

    Xmatrix = np.malmut(HtH_inverse,HtY)

    return Xmatrix

def CalculateLineOfBestFitSolution(Dataset):
    """
    Generates the H and Y matrices from the Dataset {[y_1, r_1], [y_2, r_2], ..., [y_n, r_n]}
    and calculates the line parameters [x_1 (slope), x_2 (intercept)].
    Model: y_i = x_1 * r_i + x_2
    """

    Hmatrix= []
    Ymatrix = []

    for row in Dataset: 
        y_i = row[0]
        r_i = row[1]

        # Y vector contains temperature measurements (y_i)
        Ymatrix.append([y_i])

        # H matrix row contains [r_i, 1] to solve for x_1 (slope)
        Hmatrix.append([r_i, 1.0])

    LineParam = CalculateLeastSquaresSolution(Hmatrix, Ymatrix)

    # Flatten or return the params
    return LineParam


# Weighted Least Squares Estimation

## 1. Introduction

Weighted Least Squares (WLS) is an extension of ordinary least squares that is used when different measurements have different levels of uncertainty.

Suppose the measurement model is

$$
\mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{v}
$$

where:

* $\mathbf{y}$ is the measurement vector.
* $\mathbf{H}$ is the model or design matrix.
* $\mathbf{x}$ is the unknown parameter vector that we want to estimate.
* $\mathbf{v}$ represents measurement noise or measurement errors.

The measurement noise is assumed to have covariance matrix

$$
\mathbf{R} = E[\mathbf{v}\mathbf{v}^T]
$$

where $\mathbf{R}$ describes the uncertainty of the measurements.

---

## 2. Why Weighted Least Squares?

In ordinary least squares, all measurements are treated as equally reliable.

However, in many practical problems, some measurements are more accurate than others.

For example, suppose two measurements have standard deviations

$$
\sigma_1 = 1
$$

and

$$
\sigma_2 = 3.
$$

The first measurement is more reliable because it has a smaller uncertainty.

Weighted Least Squares gives more importance to measurements with smaller uncertainty and less importance to measurements with larger uncertainty.

The weight associated with a measurement is proportional to

$$
w_i = \frac{1}{\sigma_i^2}.
$$

Therefore, a smaller value of $\sigma_i$ results in a larger weight.

---

## 3. Measurement Covariance Matrix

If the measurement errors are independent, the covariance matrix $\mathbf{R}$ is diagonal.

For $n$ measurements,

$$
\mathbf{R} =
\begin{bmatrix}
\sigma_1^2 & 0 & \cdots & 0 \
0 & \sigma_2^2 & \cdots & 0 \
\vdots & \vdots & \ddots & \vdots \
0 & 0 & \cdots & \sigma_n^2
\end{bmatrix}.
$$

The inverse covariance matrix is

$$
\mathbf{R}^{-1} =
\begin{bmatrix}
\frac{1}{\sigma_1^2} & 0 & \cdots & 0 \
0 & \frac{1}{\sigma_2^2} & \cdots & 0 \
\vdots & \vdots & \ddots & \vdots \
0 & 0 & \cdots & \frac{1}{\sigma_n^2}
\end{bmatrix}.
$$

Therefore, $\mathbf{R}^{-1}$ acts as the weighting matrix.

Measurements with smaller uncertainty receive a larger weight.

---

## 4. Weighted Least Squares Solution

The Weighted Least Squares estimate of the unknown parameter vector $\mathbf{x}$ is

$$
\hat{\mathbf{x}}
================

\left(
\mathbf{H}^T
\mathbf{R}^{-1}
\mathbf{H}
\right)^{-1}
\mathbf{H}^T
\mathbf{R}^{-1}
\mathbf{y}.
$$

Here:

* $\mathbf{H}^T$ is the transpose of $\mathbf{H}$.
* $\mathbf{R}^{-1}$ is the inverse of the measurement covariance matrix.
* $\hat{\mathbf{x}}$ is the estimated parameter vector.

The covariance matrix of the estimated parameters is

$$
\mathbf{P}
==========

\left(
\mathbf{H}^T
\mathbf{R}^{-1}
\mathbf{H}
\right)^{-1}.
$$

Therefore, the estimate can also be written as

$$
\hat{\mathbf{x}}
================

\mathbf{P}
\mathbf{H}^T
\mathbf{R}^{-1}
\mathbf{y}.
$$

---

## 5. Straight-Line Model

In this exercise, the objective is to find the line of best fit describing the relationship between temperature and RPM.

The model is

$$
y_i = x_1 r_i + x_2
$$

where:

* $y_i$ is the measured temperature.
* $r_i$ is the RPM value.
* $x_1$ is the slope of the fitted line.
* $x_2$ is the intercept of the fitted line.

The unknown parameter vector is therefore

$$
\mathbf{x}
==========

\begin{bmatrix}
x_1 \
x_2
\end{bmatrix}.
$$

For one measurement,

$$
y_i
===

\begin{bmatrix}
r_i & 1
\end{bmatrix}
\begin{bmatrix}
x_1 \
x_2
\end{bmatrix}.
$$

For $n$ measurements, the complete model becomes

$$
\begin{bmatrix}
y_1 \
y_2 \
\vdots \
y_n
\end{bmatrix}
=============

\begin{bmatrix}
r_1 & 1 \
r_2 & 1 \
\vdots & \vdots \
r_n & 1
\end{bmatrix}
\begin{bmatrix}
x_1 \
x_2
\end{bmatrix}
+
\mathbf{v}.
$$

Therefore,

$$
\mathbf{y}
==========

\begin{bmatrix}
y_1 \
y_2 \
\vdots \
y_n
\end{bmatrix}
$$

and

$$
\mathbf{H}
==========

\begin{bmatrix}
r_1 & 1 \
r_2 & 1 \
\vdots & \vdots \
r_n & 1
\end{bmatrix}.
$$

---

## 6. Dataset Structure

Each measurement in the dataset has the form

$$
[y_i,\ r_i,\ \sigma_i]
$$

where:

* $y_i$ is the measured temperature.
* $r_i$ is the RPM.
* $\sigma_i$ is the standard deviation or uncertainty associated with the temperature measurement.

For example,

```python
Dataset = [
    [65, 1, 1],
    [65, 2, 2],
    [97, 5, 1]
]
```

represents three temperature measurements.

The corresponding measurement vector is

$$
\mathbf{y}
==========

\begin{bmatrix}
65 \
65 \
97
\end{bmatrix}.
$$

The model matrix is

$$
\mathbf{H}
==========

\begin{bmatrix}
1 & 1 \
2 & 1 \
5 & 1
\end{bmatrix}.
$$

The measurement uncertainties are

$$
\sigma_1 = 1,\qquad
\sigma_2 = 2,\qquad
\sigma_3 = 1.
$$

Therefore, the covariance matrix is

$$
\mathbf{R}
==========

\begin{bmatrix}
1^2 & 0 & 0 \
0 & 2^2 & 0 \
0 & 0 & 1^2
\end{bmatrix}
=============

\begin{bmatrix}
1 & 0 & 0 \
0 & 4 & 0 \
0 & 0 & 1
\end{bmatrix}.
$$

The second measurement has a larger uncertainty and therefore contributes less weight to the final fitted line.

---

## 7. Final Line of Best Fit

After applying the Weighted Least Squares equation,

$$
\hat{\mathbf{x}}
================

\begin{bmatrix}
\hat{x}_1 \
\hat{x}_2
\end{bmatrix},
$$

where $\hat{x}_1$ is the estimated slope and $\hat{x}_2$ is the estimated intercept.

The final fitted relationship between temperature and RPM is therefore

$$
\boxed{
\hat{y} = \hat{x}_1 r + \hat{x}_2
}
$$

where $\hat{y}$ is the temperature predicted by the Weighted Least Squares model.


In [8]:
import numpy as np

def CalculateWeightedLeastSquaresSolution(Hmatrix, Rmatrix, Ymatrix):

    Hmatrix = np.array(Hmatrix)
    Rmatrix = np.array(Rmatrix)
    Ymatrix = np.array(Ymatrix)

    # Transpose of H 
    Htranspose = np.transpose(Hmatrix)

    # Inverse of R 
    Rinverse = np.linalg.inv(Rmatrix)

    # Parameter covariance matrix: 
    # P = (H^T R^-1 H)^-1 
    Pmatrix = np.linalg.inv(np.matmul(np.matmul(Htranspose, Rinverse), Hmatrix))

    # Weighted least squares solution: 
    # X = P H^T R^-1 Y 
    Xmatrix = np.matmul(np.matmul(np.matmul(Pmatrix, Htranspose), Rinverse), Ymatrix)

    return Xmatrix, Pmatrix

def CalculateLineOfBestFitSolution(Dataset):

    #Generate H, R and Y matrices 
    Hmatrix = []
    Rmatrix = []
    Ymatrix = []

    n = len(Dataset)

    for measurement in Dataset: 

        y = measurement[0]
        r = measurement[1]

        # Model: y_i = x1*r_i + x2
        Hmatrix.append(y)

        #IMPORTANT: append y, not [y]
        Ymatrix.append(y)

    # Create covariance matrix 
    Rmatrix = np.zeros((n,n))

    for i in range(n): 

        sigma = Dataset[i][2]

        # Variance = sigma^2
        Rmatrix[i][i] = sigma ** 2

    Lineparam, LineParamCov = CalculateWeightedLeastSquaresSolution(Hmatrix, Rmatrix, Ymatrix)
    return Lineparam

